In [ ]:
import logging
import matplotlib.pyplot as plt
import pandas as pd
import torch
from genkit.datasets import fetch_synthetic_data
from genkit.diffusion import DLPMEps
from genkit.flow_matching import GaussianFlowLinear
from genkit.metrics import mmd_rbf, tail_coverage_error
from genkit.nn import MLPModel
from genkit.training import train


In [ ]:
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", force=True)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print(f"[INFO] device={device} dtype={dtype}")

In [ ]:
class DLPMEpsPowerLoss(DLPMEps):
    def __init__(self, *args, loss_power=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self._loss_power = float(loss_power)
        if self._loss_power <= 0.0:
            raise ValueError(f"loss_power must be positive, got {loss_power}.")

    def _loss_fn(self, eps_hat, eps, t):
        loss_values = torch.nn.functional.mse_loss(eps_hat, eps, reduction="none")
        loss_values = loss_values.mean(dim=tuple(range(1, loss_values.ndim)))
        return loss_values.clamp_min(self._eps).pow(self._loss_power)

    @property
    def loss_power(self):
        return self._loss_power


In [ ]:
alpha = 1.7
print(f"[INFO] loading isotropic alpha-stable data alpha={alpha}")
x_train, _, x_test = fetch_synthetic_data(
    "alpha_stable",
    alpha=alpha,
    dim=2,
    n_samples=170_000,
    val_size=0.001,
    test_size=0.8,
    standardize=False,
    device=device,
    dtype=dtype,
)
n_train = x_train.shape[0]
n_test = x_test.shape[0]
print(f"[INFO] train_shape={n_train} test_shape={n_test}")


In [ ]:
loss_powers = [0.1, 0.25, 0.5]


def make_net():
    return MLPModel(dim=2, width=128, depth=3).to(device=device, dtype=dtype)


def make_models(seed):
    torch.manual_seed(seed)
    print(f"[INFO] building models seed={seed}")
    models = {
        "GF linear": GaussianFlowLinear(net=make_net(), dim=2, n_steps=128, sigma_max=1.0, device=device),
    }
    for r in loss_powers:
        models[f"DLPM r={r:g}"] = DLPMEpsPowerLoss(
            net=make_net(),
            dim=2,
            n_steps=64,
            alpha=alpha,
            n_trial_A=1,
            n_trial_G=1,
            reduce_type="mean",
            loss_power=r,
            device=device,
        )
    return models


tce_probs = {
    "TCE(90)": 1e-1,
    "TCE(99)": 1e-2,
    "TCE(99,9)": 1e-3,
    "TCE(99,99)": 1e-4,
}

@torch.no_grad()
def evaluate_model(name, model, trial, x_ref, n_tail=50_000, n_mmd=2_000):
    print(f"[INFO] evaluating trial={trial} model={name} n_tail={n_tail} n_mmd={n_mmd}")
    x_ref = x_ref[:n_tail].detach().cpu().to(torch.float64)
    x_gen = model.sample(len(x_ref)).detach().cpu().to(torch.float64)
    row = {
        "trial": trial,
        "model": name,
        "mmd_rbf": mmd_rbf(x_ref[:n_mmd], x_gen[:n_mmd]),
    }
    for label, prob in tce_probs.items():
        probs = torch.tensor([prob], dtype=torch.float64)
        row[label] = tail_coverage_error(x_ref, x_gen, probs=probs, tail="upper", mode="log")
    return row


In [ ]:
n_tail = n_test
n_mmd = n_test // 10
n_trials = 5
train_kwargs = dict(batch_size=128, n_epochs=32, lr=5e-4, device=device, use_adamw=False, lr_schedule="constant", freq_logging=8)

In [ ]:
print(f"[INFO] n_trials={n_trials} loss_powers={loss_powers} train_kwargs={train_kwargs}")

rows = []
for trial in range(1, n_trials + 1):

    print(f"[INFO] trial {trial}/{n_trials}: start")
    models = make_models(seed=trial - 1)

    for name, model in models.items():
        print(f"[INFO] trial {trial}/{n_trials}: train {name}")
        model, _ = train(model, x_train, **train_kwargs)

        print(f"[INFO] trial {trial}/{n_trials}: evaluate {name}")
        rows.append(evaluate_model(name, model, trial, x_test, n_tail=n_tail, n_mmd=n_mmd))

    print(f"[INFO] trial {trial}/{n_trials}: done")


In [ ]:
print("[INFO] computing test-vs-test reference metrics")

x_ref = x_test.detach().cpu().to(torch.float64)

for trial in range(1, n_trials + 1):
    generator = torch.Generator(device="cpu").manual_seed(10_000 + trial)
    perm = torch.randperm(x_ref.shape[0], generator=generator)

    x_a = x_ref[perm[:n_tail]]
    x_b = x_ref[perm[n_tail: 2 * n_tail]]

    row = {
        "trial": trial,
        "model": "test vs test",
        "mmd_rbf": mmd_rbf(x_a[:n_mmd], x_b[:n_mmd]),
    }

    for label, prob in tce_probs.items():
        probs = torch.tensor([prob], dtype=torch.float64)
        row[label] = tail_coverage_error(x_a, x_b, probs=probs, tail="upper", mode="log")

    rows.append(row)

In [ ]:
print("[INFO] aggregating results")
results = pd.DataFrame(rows)
tce_cols = ["TCE(90)", "TCE(99)", "TCE(99,9)", "TCE(99,99)"]
tce_x = [0.90, 0.99, 0.999, 0.9999]

grouped = results.groupby("model")
mean_results = grouped[["mmd_rbf", *tce_cols]].mean()
std_results = grouped[tce_cols].std().fillna(0.0)

fig, ax = plt.subplots(figsize=(6, 5))

for model_name, row in mean_results.iterrows():
    y = row[tce_cols].to_numpy(dtype=float)
    yerr = std_results.loc[model_name, tce_cols].to_numpy(dtype=float)
    lower = (y - yerr).clip(min=1e-12)
    upper = y + yerr
    label = f"{model_name} (MMD-RBF={row['mmd_rbf']:.4g})"
    line, = ax.plot(tce_x, y, marker="o", linewidth=2, label=label, alpha=0.5)
    ax.fill_between(tce_x, lower, upper, color=line.get_color(), alpha=0.2)

ax.set_xscale("logit")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-2)
ax.set_xticks(tce_x)
ax.set_xticklabels(["90", "99", "99.9", "99.99"])
ax.set_xlabel("tail quantile (%)", fontsize=12)
ax.set_ylabel("TCE, upper tail log error", fontsize=12)

ax.grid(True, which="both", alpha=0.4)
ax.legend()

plt.tight_layout()
